[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/03_classification/03_classification.ipynb)

# 03. Classification — Logistic & Softmax Regression

> 원본 강의: [Lec 5–6, 모두를 위한 머신러닝과 딥러닝](https://hunkim.github.io/ml/) — Logistic Regression, Softmax Regression(다중 분류)

## 이론

### 1) 왜 Linear Regression으로 분류를 못할까
분류 문제(예: 합격/불합격)에 직선을 그대로 쓰면 출력이 0~1을 벗어나고, outlier 하나에 결정 경계가 크게 흔들립니다. 그래서 출력을 0~1 사이 **확률**로 눌러주는 함수가 필요합니다.

### 2) Logistic Regression (이진 분류)
Sigmoid 함수로 선형 결합값을 확률로 변환합니다.
$$H(X) = \frac{1}{1 + e^{-(XW + b)}}$$
Cost로는 MSE 대신 **Cross-Entropy**를 사용합니다 (MSE를 쓰면 non-convex라 local minima에 빠지기 쉬움).
$$J(W) = -\frac{1}{m}\sum_{i=1}^{m} \Big[ y^{(i)} \log H(x^{(i)}) + (1-y^{(i)}) \log(1-H(x^{(i)})) \Big]$$

### 3) Softmax Regression (다중 클래스 분류)
클래스가 3개 이상일 때는 각 클래스에 대한 점수(logit)를 확률 분포로 바꿔주는 **Softmax**를 사용합니다.
$$S(y_i) = \frac{e^{y_i}}{\sum_j e^{y_j}}$$
Cost는 **Cross-Entropy**를 다중 클래스로 확장한 형태입니다.
$$J(W) = -\frac{1}{m}\sum_{i=1}^{m}\sum_{k=1}^{K} y_k^{(i)} \log S(y_k^{(i)})$$
$y_k^{(i)}$는 one-hot 인코딩된 실제 라벨입니다.

## 실습 0. Colab 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)
if IN_COLAB:
    !pip install -q scikit-learn numpy matplotlib

## 실습 1. Logistic Regression을 numpy로 직접 구현 (이진 분류)

공부 시간에 따른 시험 합격 여부를 Sigmoid + Cross-Entropy + Gradient Descent로 직접 학습합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
hours = np.linspace(0, 10, 60)
prob_pass = 1 / (1 + np.exp(-(hours - 5)))
passed = (rng.random(len(hours)) < prob_pass).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

W, b = 0.0, 0.0
lr = 0.1
epochs = 500
m = len(hours)
cost_history = []

for epoch in range(epochs):
    z = W * hours + b
    h = sigmoid(z)
    eps = 1e-7  # log(0) 방지
    cost = -np.mean(passed * np.log(h + eps) + (1 - passed) * np.log(1 - h + eps))
    cost_history.append(cost)

    dW = np.mean((h - passed) * hours)
    db = np.mean(h - passed)
    W -= lr * dW
    b -= lr * db

    if epoch % 100 == 0:
        print(f"epoch {epoch:3d}  cost={cost:.4f}")

print(f"\n최종 W={W:.3f}, b={b:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(cost_history)
axes[0].set_title("Cross-Entropy Cost")
axes[0].set_xlabel("epoch")

xs = np.linspace(0, 10, 200)
axes[1].scatter(hours, passed, alpha=0.6, label="data (0/1)")
axes[1].plot(xs, sigmoid(W * xs + b), color="red", label="학습된 확률")
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Logistic Regression 결과")
axes[1].legend()
plt.show()

## 실습 2. Softmax Regression — Iris 3-클래스 분류 (scikit-learn)

직접 구현한 원리를 이제 scikit-learn의 `LogisticRegression(multi_class="multinomial")`으로 실무 방식으로 적용해봅니다 (내부적으로 Softmax + Cross-Entropy를 사용).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=0, stratify=iris.target
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

softmax_clf = LogisticRegression(max_iter=200)  # 다중 클래스 -> 자동으로 softmax(multinomial) 사용
softmax_clf.fit(X_train_s, y_train)
y_pred = softmax_clf.predict(X_test_s)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# 클래스별 확률(softmax 출력)을 직접 확인
sample = X_test_s[:5]
probs = softmax_clf.predict_proba(sample)
for i, p in enumerate(probs):
    print(f"샘플 {i}: {dict(zip(iris.target_names, np.round(p, 3)))}")

## 정리 & 연습 문제
- 실습 1에서 `lr`을 바꿔가며 Cross-Entropy Cost 수렴 속도를 비교해보세요.
- 실습 2에서 `StandardScaler`를 빼고 학습하면 성능이 어떻게 달라지는지 확인해보세요.
- 다음 노트북([04_neural_networks.ipynb](../04_neural_networks/04_neural_networks.ipynb))에서는 Logistic Regression(=퍼셉트론 1개)으로 풀 수 없는 XOR 문제를 신경망으로 풀어봅니다.


**해설/정답**: [03_classification_solutions.ipynb](03_classification_solutions.ipynb)
